# MSCS 634 - Lab 1: Data Visualization, Preprocessing and Statistical Analysis

**Name:** Fortunato Hernandez
**Course:** MSCS 634
**Assignment:** Lab 1

---
## Step 1: Data Collection

We generate a synthetic retail sales dataset spanning two years (2023-2024) across multiple product categories, regions, and customer segments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

n = 500
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Books']
regions     = ['North', 'South', 'East', 'West']
segments    = ['Consumer', 'Corporate', 'Home Office']

dates = pd.date_range('2023-01-01', '2024-12-31', periods=n)

df = pd.DataFrame({
    'Order_Date':       dates,
    'Category':         np.random.choice(categories, n),
    'Region':           np.random.choice(regions, n),
    'Customer_Segment': np.random.choice(segments, n),
    'Quantity':         np.random.randint(1, 20, n),
    'Unit_Price':       np.round(np.random.uniform(5, 500, n), 2),
    'Discount':         np.round(np.random.uniform(0, 0.4, n), 2),
    'Shipping_Cost':    np.round(np.random.uniform(2, 50, n), 2),
    'Customer_Rating':  np.round(np.random.uniform(1, 5, n), 1),
})

df['Revenue'] = np.round(df['Quantity'] * df['Unit_Price'] * (1 - df['Discount']), 2)
df['Profit']  = np.round(df['Revenue'] - df['Shipping_Cost'] - (df['Quantity'] * df['Unit_Price'] * 0.6), 2)

# Inject ~5% missing values
for col in ['Discount', 'Customer_Rating', 'Shipping_Cost']:
    df.loc[df.sample(frac=0.05, random_state=42).index, col] = np.nan

# Inject outliers in Revenue
outlier_idx = np.random.choice(df.index, 8, replace=False)
df.loc[outlier_idx, 'Revenue'] = df['Revenue'].mean() + np.random.uniform(4, 6, 8) * df['Revenue'].std()

df.to_csv('retail_sales.csv', index=False)
print(f"Dataset shape: {df.shape}")
df.head()

---
## Step 2: Data Visualization

### 2.1 Scatter Plot - Revenue vs Quantity (colored by Category)

In [ ]:
plt.figure(figsize=(10, 6))
for cat in df['Category'].unique():
    subset = df[df['Category'] == cat]
    plt.scatter(subset['Quantity'], subset['Revenue'], alpha=0.6, label=cat, edgecolors='w', linewidth=0.4)
plt.xlabel('Quantity Sold')
plt.ylabel('Revenue ($)')
plt.title('Revenue vs. Quantity by Product Category')
plt.legend(title='Category')
plt.tight_layout()
plt.show()

**Insight:** Revenue increases with quantity, but the slope varies by category. Electronics transactions cluster at higher revenue for any given quantity (higher unit prices), while Books stay in the lower-revenue band.

### 2.2 Line Plot - Monthly Revenue Trend

In [ ]:
monthly = df.set_index('Order_Date').resample('ME')['Revenue'].sum().reset_index()
monthly.columns = ['Month', 'Total_Revenue']

plt.figure(figsize=(10, 5))
plt.plot(monthly['Month'], monthly['Total_Revenue'], marker='o', linewidth=2, color='steelblue')
plt.fill_between(monthly['Month'], monthly['Total_Revenue'], alpha=0.15, color='steelblue')
plt.xlabel('Month')
plt.ylabel('Total Revenue ($)')
plt.title('Monthly Revenue Trend (2023-2024)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Insight:** Revenue fluctuates month-to-month without a clear long-term trend. Peaks likely correspond to months with more high-value orders.

### 2.3 Bar Chart - Average Revenue by Region

In [ ]:
region_rev = df.groupby('Region')['Revenue'].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
bars = plt.bar(region_rev.index, region_rev.values, color=['#4C72B0','#55A868','#C44E52','#8172B2'])
plt.ylabel('Average Revenue ($)')
plt.title('Average Revenue by Region')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'${bar.get_height():,.0f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

**Insight:** The four regions have broadly comparable average revenues. Minor differences are attributable to random variation.

### 2.4 Histogram - Distribution of Customer Ratings

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['Customer_Rating'].dropna(), bins=20, color='teal', edgecolor='white', alpha=0.85)
plt.xlabel('Customer Rating')
plt.ylabel('Frequency')
plt.title('Distribution of Customer Ratings')
plt.tight_layout()
plt.show()

**Insight:** Ratings are approximately uniformly distributed between 1 and 5, with no obvious skew.

### 2.5 Box Plot - Revenue Distribution by Category

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Category', y='Revenue', palette='Set2')
plt.title('Revenue Distribution by Product Category')
plt.ylabel('Revenue ($)')
plt.tight_layout()
plt.show()

**Insight:** All categories show right-skewed revenue distributions with several outliers on the upper end.

### 2.6 Pie Chart - Order Proportion by Customer Segment

In [ ]:
seg_counts = df['Customer_Segment'].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(seg_counts, labels=seg_counts.index, autopct='%1.1f%%',
        startangle=140, colors=['#66b3ff','#99ff99','#ffcc99'],
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
plt.title('Order Distribution by Customer Segment')
plt.tight_layout()
plt.show()

**Insight:** Orders are distributed roughly equally among Consumer, Corporate, and Home Office segments.

---
## Step 3: Data Preprocessing

### 3.1 Handling Missing Values

In [ ]:
print("Missing values BEFORE handling:")
print(df.isnull().sum())
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

In [ ]:
# Discount: fill with median
df['Discount'] = df['Discount'].fillna(df['Discount'].median())

# Customer_Rating: forward fill then backward fill
df.sort_values('Order_Date', inplace=True)
df['Customer_Rating'] = df['Customer_Rating'].ffill()
df['Customer_Rating'] = df['Customer_Rating'].bfill()

# Shipping_Cost: fill with mean
df['Shipping_Cost'] = df['Shipping_Cost'].fillna(df['Shipping_Cost'].mean())

# Recalculate derived columns
df['Revenue'] = np.round(df['Quantity'] * df['Unit_Price'] * (1 - df['Discount']), 2)
df['Profit']  = np.round(df['Revenue'] - df['Shipping_Cost'] - (df['Quantity'] * df['Unit_Price'] * 0.6), 2)

print("Missing values AFTER handling:")
print(df.isnull().sum())

### 3.2 Outlier Detection and Removal (IQR Method)

In [ ]:
Q1  = df['Revenue'].quantile(0.25)
Q3  = df['Revenue'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1 = {Q1:.2f},  Q3 = {Q3:.2f},  IQR = {IQR:.2f}")
print(f"Lower bound = {lower_bound:.2f},  Upper bound = {upper_bound:.2f}")

outliers = df[(df['Revenue'] < lower_bound) | (df['Revenue'] > upper_bound)]
print(f"\nNumber of outliers detected: {len(outliers)}")
print("\nOutlier rows (Revenue):")
outliers[['Order_Date', 'Category', 'Quantity', 'Unit_Price', 'Revenue']].head(10)

In [ ]:
print(f"Shape BEFORE outlier removal: {df.shape}")
df_clean = df[(df['Revenue'] >= lower_bound) & (df['Revenue'] <= upper_bound)].copy()
print(f"Shape AFTER  outlier removal: {df_clean.shape}")
print(f"Rows removed: {len(df) - len(df_clean)}")

### 3.3 Data Reduction

In [ ]:
print(f"Shape BEFORE reduction: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")

In [ ]:
# Dimension elimination - drop less relevant columns
df_reduced = df_clean.drop(columns=['Shipping_Cost', 'Customer_Rating'])

# Sampling - take a 60% random sample
df_reduced = df_reduced.sample(frac=0.60, random_state=42)

print(f"Shape AFTER  reduction: {df_reduced.shape}")
print(f"Columns: {list(df_reduced.columns)}")
df_reduced.head()

### 3.4 Data Scaling and Discretization

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

numeric_cols = ['Quantity', 'Unit_Price', 'Revenue', 'Profit']

print("BEFORE scaling (first 5 rows):")
print(df_reduced[numeric_cols].head().to_string())

In [ ]:
# Min-Max Scaling
scaler_mm = MinMaxScaler()
df_minmax = df_reduced.copy()
df_minmax[numeric_cols] = scaler_mm.fit_transform(df_reduced[numeric_cols])

print("AFTER Min-Max Scaling (first 5 rows):")
print(df_minmax[numeric_cols].head().to_string())

In [ ]:
# Z-score Standardization
scaler_z = StandardScaler()
df_zscore = df_reduced.copy()
df_zscore[numeric_cols] = scaler_z.fit_transform(df_reduced[numeric_cols])

print("AFTER Z-score Standardization (first 5 rows):")
print(df_zscore[numeric_cols].head().to_string())

In [ ]:
# Discretization - bin Revenue into categories
df_disc = df_reduced.copy()
df_disc['Revenue_Tier'] = pd.cut(df_disc['Revenue'], bins=3, labels=['Low', 'Medium', 'High'])

print("Revenue discretized into tiers:")
print(df_disc[['Revenue', 'Revenue_Tier']].head(10).to_string())
print(f"\nTier value counts:\n{df_disc['Revenue_Tier'].value_counts()}")

---
## Step 4: Statistical Analysis

We use the cleaned (unscaled) dataset df_clean for meaningful statistics.

### 4.1 General Overview

In [ ]:
df_clean.info()

In [ ]:
df_clean.describe()

### 4.2 Central Tendency Measures

In [ ]:
numeric = df_clean.select_dtypes(include='number')

central = pd.DataFrame({
    'Min':    numeric.min(),
    'Max':    numeric.max(),
    'Mean':   numeric.mean(),
    'Median': numeric.median(),
    'Mode':   numeric.mode().iloc[0]
})
central.round(2)

### 4.3 Dispersion Measures

In [ ]:
dispersion = pd.DataFrame({
    'Range':     numeric.max() - numeric.min(),
    'Q1':        numeric.quantile(0.25),
    'Q3':        numeric.quantile(0.75),
    'IQR':       numeric.quantile(0.75) - numeric.quantile(0.25),
    'Variance':  numeric.var(),
    'Std_Dev':   numeric.std()
})
dispersion.round(2)

### 4.4 Correlation Analysis

In [ ]:
corr_matrix = numeric.corr().round(3)
print("Correlation Matrix:")
corr_matrix

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            linewidths=0.5, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

**Key correlation insights:**
- Revenue and Quantity show a moderate positive correlation.
- Revenue and Unit_Price are also positively correlated.
- Profit is strongly correlated with Revenue, which is expected since profit is derived from revenue.
- Discount shows a slight negative correlation with Revenue.

---
## Summary

| Step | Key Takeaway |
|------|-------------|
| Data Collection | 500-row synthetic retail sales dataset with 11 features |
| Visualization | Six chart types revealed revenue distributions, category patterns, and temporal trends |
| Preprocessing | Imputed ~5% missing values, removed IQR-based outliers, reduced dimensions and sampled 60% |
| Statistical Analysis | Computed central tendency, dispersion, and correlations; Revenue-Quantity and Revenue-Profit show strongest relationships |